# Computational Theory — SHA-256

## Introduction

This notebook explores the SHA-256 cryptographic hash function through a step-by-step implementation based on the Secure Hash Standard FIPS 180-4 ([see official documentation](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)).

The objective is to investigate the underlying data representations, implement the mathematical and bitwise operations, construct the compression function, and validate the completed implementation against Python's built-in hashing functionality ([see Python's hashlib documentation](https://docs.python.org/3/library/hashlib.html)).

## Problem 1 — Representing SHA-256 Data

### 1.1 Introduction

SHA-256 operates on messages using fixed-size blocks and 32-bit words. Before implementing the algorithm, it is important to understand how these values can be represented in Python and how the chosen representations support the required operations ([see the Secure Hash Standard, FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)).

This section investigates representations for unsigned 32-bit words, sequences of words, input messages, message blocks, hash values, and big-endian integers.

### 1.2 Representing a 32-bit Word

A 32-bit word consists of 32 binary digits (bits). Each bit can hold a value of either zero or one. SHA-256 uses 32-bit words throughout its internal processing, so values must be represented within this fixed width.

An unsigned 32-bit word can represent values from zero to $2^{32}-1$, inclusive. This gives a total of $2^{32}$ possible values.

For example, the decimal value 42 can be represented as a 32-bit binary word by padding its binary representation with leading zeros. Fixed width representations ensure that each word contains exactly 32 bits, as required by SHA-256 ([see the Secure Hash Standard, FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)).


In [9]:
# Represent the decimal value 42 using exactly 32 binary bits.
value = 42
binary_word = f"{value:032b}"

print(f"Decimal value: {value}")
print(f"Binary representation: {binary_word}")
print(f"Number of bits: {len(binary_word)}")

Decimal value: 42
Binary representation: 00000000000000000000000000101010
Number of bits: 32


In [10]:
# Verify that the binary representation contains exactly 32 bits.
assert len(binary_word) == 32

# Verify that the binary representation corresponds to the original value.
assert int(binary_word, 2) == value

print("All 32-bit representation tests passed.")

All 32-bit representation tests passed.


### 1.3 Representing Sequences of 32-bit Words

SHA-256 processes data using groups of 32-bit words. A sequence of words needs to preserve both the individual values and their order.

Python lists provide a straightforward way to store an ordered collection of integers. Elements can be accessed by their position, making lists suitable for demonstrating sequences of words ([see official Python documentation on lists](https://docs.python.org/3/tutorial/introduction.html#lists)).

A list will be used to explore how multiple words can be stored and accessed in sequence.

In [11]:
# Store four example words in an ordered sequence.
words = [42, 100, 256, 1024]

print(f"Words: {words}")
print(f"Number of words: {len(words)}")
print(f"First word: {words[0]}")
print(f"Last word: {words[-1]}")

Words: [42, 100, 256, 1024]
Number of words: 4
First word: 42
Last word: 1024


In [12]:
# Verify the sequence contains the expected values and number of words.
assert len(words) == 4
assert words[0] == 42
assert words[-1] == 1024
assert words == [42, 100, 256, 1024]

print("All word sequence tests passed.")

All word sequence tests passed.


#### Observations

The experiment demonstrates that Python lists can store multiple values in an ordered sequence. The number of elements can be determined using `len()`, and individual elements can be accessed using their index. This provides a simple representation for storing multiple words in order, which is useful when working with groups of 32-bit words in SHA-256.

### 1.4 Representing Input Messages

SHA-256 accepts input messages as bytes rather than directly as Python strings. A Python string represents text, while a `bytes` object represents a sequence of byte values. Text can be converted into bytes using an encoding such as UTF-8.

Python's `hashlib` needs bytes like data when providing input to a hash function, so messages must be encoded before they can be processed ([see official Python documentation on hashlib](https://docs.python.org/3/library/hashlib.html)).

UTF-8 encoding will be used to convert text messages into bytes for this investigation.

In [13]:
# Convert a text message into bytes using UTF-8.
message = "Hello world, SHA-256"
encoded_message = message.encode("utf-8")

print(f"Message: {message}")
print(f"Type: {type(encoded_message)}")
print(f"Bytes: {encoded_message}")
print(f"Length: {len(encoded_message)}")

Message: Hello world, SHA-256
Type: <class 'bytes'>
Bytes: b'Hello world, SHA-256'
Length: 20


In [14]:
# Verify that the message was encoded as bytes.
assert isinstance(encoded_message, bytes)

# Verify that decoding restores the original message.
assert encoded_message.decode("utf-8") == message

print("All message representation tests passed.")

All message representation tests passed.


#### Observations

The experiment demonstrates that a Python string can be encoded into a bytes object using UTF-8. The encoded message can then be accessed as a sequence of bytes, and decoding it back using the same encoding restores the original text.

This confirms that message input can be represented as bytes before being processed by SHA-256.

### 1.5 Representing a 512-bit Message Block

SHA-256 processes messages in blocks of 512 bits, equivalent to 64 bytes. Each byte contains 8 bits, so a block contains 64 bytes. The fixed block size is defined by the SHA-256 standard ([see the Secure Hash Standard, FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf)).

Python's `bytes` type can be used to represent a block as a sequence of byte values. This allows the block size to be measured and individual bytes to be accessed.

An example 64-byte block will be created to investigate this representation.

In [ ]:
# Create an example block containing 64 bytes.
block = bytes(range(64))

print(f"Block: {block}")
print(f"Block size: {len(block)} bytes")
print(f"Block size in bits: {len(block) * 8}")
print(f"First byte: {block[0]}")
print(f"Last byte: {block[-1]}")

In [ ]:
# Verify that the block contains exactly 64 bytes.
assert len(block) == 64

# Verify that the block represents exactly 512 bits.
assert len(block) * 8 == 512

# Verify that the bytes are stored in the expected sequence.
assert block[0] == 0
assert block[-1] == 63

print("All message block representation tests passed.")

#### Observations

The experiment demonstrates that a Python `bytes` object can store a sequence of 64 bytes, representing a total of 512 bits. The individual bytes can be accessed by their positions within the block.

This provides a suitable representation for SHA-256 message blocks, as the algorithm processes data in fixed size 512-bit blocks.